In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/activities_final.csv", parse_dates=["date"])

# Dataset 1 — all runs, no Whoop features
strava_features = ["distance_km", "duration_sec", "elevation_m",
                   "avg_hr", "avg_speed", "pace_min_per_km",
                   "rolling_7d_km_daily", "rolling_28d_km_daily", "rolling_42d_km_daily",
                   "acwr_daily", "day_of_week", "month", "hour"]

df_strava = df[strava_features + ["date", "id"]].copy()
print(f"Strava dataset: {len(df_strava)} runs")

# Dataset 2 — runs with Whoop data
whoop_features = strava_features + ["recovery_score", "hrv_rmssd",
                                     "resting_hr", "sleep_performance_pct",
                                     "deep_sleep_min", "rem_sleep_min"]

df_whoop = df[df["recovery_score"].notna()][whoop_features + ["date", "id"]].copy()
print(f"Whoop dataset: {len(df_whoop)} runs")

Strava dataset: 509 runs
Whoop dataset: 235 runs


In [2]:
print(df.columns.tolist())

['date', 'id', 'name', 'distance_km', 'duration_sec', 'elevation_m', 'avg_hr', 'max_hr', 'avg_speed', 'pace_min_per_km', 'week', 'day_of_week', 'month', 'hour', 'time_of_day', 'days_since_last_run', 'rolling_7d_km', 'rolling_28d_km', 'rolling_42d_km', 'acwr', 'rolling_7d_km_daily', 'rolling_28d_km_daily', 'rolling_42d_km_daily', 'acwr_daily', 'date_day', 'recovery_score', 'hrv_rmssd', 'resting_hr', 'skin_temp_celsius', 'spo2_pct', 'sleep_performance_pct', 'sleep_duration_min', 'deep_sleep_min', 'rem_sleep_min', 'light_sleep_min']


## Model 1 — XGBoost Pace Predictor
Predicting average pace (min/km) from training load and activity features.
Baseline model — establishes performance ceiling for more complex models.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import numpy as np

# Target: pace
# Drop rows where target is missing
model1_features = ["distance_km", "elevation_m", "rolling_7d_km_daily", 
                   "rolling_28d_km_daily", "acwr_daily", 
                   "day_of_week", "month", "hour"]

df_model1 = df_strava[model1_features + ["pace_min_per_km"]].dropna()

X = df_model1[model1_features]
y = df_model1["pace_min_per_km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Training samples: 402
Test samples: 101


In [4]:
# Filter to easy runs only (avg_hr < 155 or no HR — use distance as proxy)
df_model1 = df_strava[model1_features + ["pace_min_per_km", "avg_hr"]].dropna(
    subset=model1_features + ["pace_min_per_km"]
)

# Only easy runs where HR is available and below 155
df_model1_easy = df_model1[
    (df_model1["avg_hr"].isna()) | (df_model1["avg_hr"] < 155)
]

print(f"Easy runs for modelling: {len(df_model1_easy)}")

X = df_model1_easy[model1_features]
y = df_model1_easy["pace_min_per_km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Easy runs for modelling: 419


In [5]:
model1_features = ["distance_km", "elevation_m", "rolling_7d_km_daily",
                   "rolling_28d_km_daily", "acwr_daily",
                   "day_of_week", "month", "hour", "avg_hr"]  # add avg_hr

df_model1 = df_strava[model1_features + ["pace_min_per_km"]].dropna()

X = df_model1[model1_features]
y = df_model1["pace_min_per_km"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.3f} min/km")
print(f"R²: {r2:.3f}")
print(f"Mean error in seconds/km: {mae * 60:.1f}s")

MAE: 0.353 min/km
R²: 0.472
Mean error in seconds/km: 21.2s


In [7]:
import plotly.express as px
importance = pd.DataFrame({
    "feature": model1_features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=True)

fig = px.bar(importance, x="importance", y="feature", 
             orientation="h",
             title="XGBoost Feature Importance — Pace Predictor")
fig.show()

### Model 1 Results
- **R² = 0.472** — model explains 47% of pace variance on easy runs
- **MAE = 21.2 sec/km** — average prediction error
- **Key finding:** Heart rate is the dominant predictor, followed by chronic training load and elevation
- **Limitation:** Model only valid for easy runs (HR < 155bpm)

## Model 3 — HR Efficiency Trend Forecasting (Holt-Winters Exponential Smoothing)
Modelling the temporal trend of cardiovascular fitness using HR efficiency on easy runs.
Holt-Winters chosen over LSTM due to dataset size (60 weekly observations) — 
exponential smoothing with trend component is more statistically appropriate 
than a neural net for short univariate time series.
Note: LSTM architecture was prototyped but requires more data to generalise reliably.

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Load data fresh
df = pd.read_csv("../data/processed/activities_final.csv", parse_dates=["date"])
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)

# Easy runs
df_easy = df[
    (df["avg_hr"].notna()) & 
    (df["avg_hr"] < 155)
].copy()

df_easy["speed_m_per_min"] = (df_easy["distance_km"] * 1000) / (df_easy["duration_sec"] / 60)
df_easy["hr_efficiency"] = df_easy["speed_m_per_min"] / df_easy["avg_hr"]
df_easy = df_easy[df_easy["hr_efficiency"] < 1.8].copy()
df_easy = df_easy.sort_values("date").reset_index(drop=True)

# Weekly aggregation
df_easy["week"] = pd.to_datetime(df_easy["date"]).dt.to_period("W").dt.start_time
weekly = df_easy.groupby("week")["hr_efficiency"].mean().reset_index()
weekly.columns = ["week", "hr_efficiency"]
weekly = weekly.dropna().reset_index(drop=True)

print(f"Weekly data points: {len(weekly)}")

# Train/test split — last 8 weeks as test
train = weekly["hr_efficiency"][:-8]
test = weekly["hr_efficiency"][-8:]

# Fit Holt-Winters
model_hw = ExponentialSmoothing(
    train,
    trend="add",
    seasonal=None,
    initialization_method="estimated"
).fit()

# Forecast
forecast = model_hw.forecast(8)

mae = np.mean(np.abs(forecast.values - test.values))
print(f"MAE: {mae:.4f} m/min/bpm")

import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly["week"].astype(str), y=weekly["hr_efficiency"], name="Actual"))
fig.add_trace(go.Scatter(x=weekly["week"].iloc[-8:].astype(str), y=forecast.values, name="Forecast"))
fig.update_layout(title="HR Efficiency Forecast vs Actual")
fig.show()

Weekly data points: 68
MAE: 0.1205 m/min/bpm


In [11]:
# Remove outlier properly
weekly = weekly[weekly["hr_efficiency"] < 1.5].reset_index(drop=True)

# Train/test split
train = weekly["hr_efficiency"][:-16]
test = weekly["hr_efficiency"][-16:]
test_weeks = weekly["week"][-16:]

# Fit Holt-Winters
model_hw = ExponentialSmoothing(
    train,
    trend="add",
    seasonal=None,
    initialization_method="estimated"
).fit()

# Forecast
forecast = model_hw.forecast(16)

mae = np.mean(np.abs(forecast.values - test.values))
print(f"MAE: {mae:.4f} m/min/bpm")

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=weekly["week"].astype(str), 
    y=weekly["hr_efficiency"], 
    name="Actual"))
fig.add_trace(go.Scatter(
    x=test_weeks.astype(str),   # use actual dates not indices
    y=forecast.values, 
    name="Forecast",
    line=dict(color="red", dash="dash")))
fig.update_layout(title="HR Efficiency Forecast vs Actual (easy runs)")
fig.show()

MAE: 0.0891 m/min/bpm


## Model 5 — Personalised Readiness Score
Building our own readiness model from Whoop biometrics and training load,
then comparing it against Whoop's black-box recovery score.
Target: next-day pace deviation from baseline (did you run faster or slower than expected?)
This is the most direct parallel to what SquadAssist does — predicting athlete
readiness from physiological signals.

In [12]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

# Load final dataset
df = pd.read_csv("../data/processed/activities_final.csv", parse_dates=["date"])
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)
df = df.sort_values("date").reset_index(drop=True)

# Easy runs with Whoop data only
df_ready = df[
    (df["avg_hr"].notna()) &
    (df["avg_hr"] < 155) &
    (df["recovery_score"].notna())
].copy()

# Compute HR efficiency as performance proxy
df_ready["speed_m_per_min"] = (df_ready["distance_km"] * 1000) / (df_ready["duration_sec"] / 60)
df_ready["hr_efficiency"] = df_ready["speed_m_per_min"] / df_ready["avg_hr"]
df_ready = df_ready[df_ready["hr_efficiency"] < 1.5].copy()

# Compute baseline efficiency (rolling 4-week median)
df_ready = df_ready.set_index("date")
df_ready["baseline_efficiency"] = df_ready["hr_efficiency"].rolling("28D").median()
df_ready = df_ready.reset_index()

# Target: deviation from baseline
# Positive = ran better than expected, Negative = ran worse
df_ready["performance_delta"] = df_ready["hr_efficiency"] - df_ready["baseline_efficiency"]

print(f"Runs available: {len(df_ready)}")
df_ready[["date", "recovery_score", "hrv_rmssd", "hr_efficiency", 
          "baseline_efficiency", "performance_delta"]].head(10)

Runs available: 173


,date,recovery_score,hrv_rmssd,hr_efficiency,baseline_efficiency,performance_delta
0,2024-12-11 18:35:40,57.0,31.933506,1.348380,1.348380,0.000000
1,2024-12-12 11:04:37,51.0,24.478584,1.186765,1.267573,-0.080807
2,2024-12-25 10:42:25,62.0,24.408240,1.189933,1.189933,0.000000
3,2024-12-30 12:55:20,55.0,22.568815,1.270903,1.230418,0.040485
4,2025-01-14 19:31:32,75.0,26.740753,1.033035,1.189933,-0.156899
5,2025-01-21 18:39:01,57.0,24.404726,1.101754,1.145844,-0.044090
6,2025-01-22 18:56:38,48.0,23.033817,1.106888,1.104321,0.002567
7,2025-01-26 17:34:14,32.0,20.405819,1.040288,1.101754,-0.061466
8,2025-01-27 18:59:33,53.0,22.184809,1.139600,1.101754,0.037846
9,2025-01-29 19:37:04,89.0,31.602450,1.157969,1.104321,0.053648


In [13]:
readiness_features = [
    "recovery_score",        # Whoop's own score
    "hrv_rmssd",            # HRV
    "resting_hr",           # resting HR
    "sleep_performance_pct", # sleep quality
    "deep_sleep_min",       # deep sleep
    "rem_sleep_min",        # REM sleep
    "rolling_7d_km_daily",  # recent training load
    "acwr_daily",           # acute:chronic ratio
    "days_since_last_run",  # rest days
]

df_model5 = df_ready[readiness_features + ["performance_delta"]].dropna()
print(f"Training samples: {len(df_model5)}")

X = df_model5[readiness_features]
y = df_model5["performance_delta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_readiness = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

model_readiness.fit(X_train, y_train)
y_pred = model_readiness.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.4f} m/min/bpm")
print(f"R²: {r2:.3f}")

Training samples: 173
MAE: 0.0534 m/min/bpm
R²: -0.170


In [14]:
importance = pd.DataFrame({
    "feature": readiness_features,
    "importance": model_readiness.feature_importances_
}).sort_values("importance", ascending=True)

fig = px.bar(importance, x="importance", y="feature",
             orientation="h",
             title="Readiness Model — Feature Importance")
fig.show()

In [15]:
# How well does Whoop's own recovery_score predict performance delta?
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train[["recovery_score"]], y_train)
y_pred_whoop = lr.predict(X_test[["recovery_score"]])

mae_whoop = mean_absolute_error(y_test, y_pred_whoop)
r2_whoop = r2_score(y_test, y_pred_whoop)

print("--- Our Model (XGBoost, all features) ---")
print(f"MAE: {mae:.4f} | R²: {r2:.3f}")
print()
print("--- Whoop Score Only (linear regression) ---")
print(f"MAE: {mae_whoop:.4f} | R²: {r2_whoop:.3f}")
print()
if r2 > r2_whoop:
    print("✅ Our model outperforms Whoop's black-box score")
else:
    print("❌ Whoop's score is stronger — interesting finding!")

--- Our Model (XGBoost, all features) ---
MAE: 0.0534 | R²: -0.170

--- Whoop Score Only (linear regression) ---
MAE: 0.0504 | R²: -0.074

❌ Whoop's score is stronger — interesting finding!


## Model 2 — Race Time Predictor (Weekly XGBoost)
Predicting next week's average easy pace from training load and biometric features.
Reframed from race time prediction due to limited race labels (only ~5 actual races).
Weekly aggregation reduces noise vs run-level prediction.

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

# Load data
df = pd.read_csv("../data/processed/activities_final.csv", parse_dates=["date"])
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)
df = df.sort_values("date").reset_index(drop=True)

# Easy runs only
df_easy = df[
    (df["avg_hr"].notna()) &
    (df["avg_hr"] < 155) &
    (df["pace_min_per_km"] <= 8)
].copy()


# Easy runs only
df_easy = df[
    (df["avg_hr"].notna()) &
    (df["avg_hr"] < 155) &
    (df["pace_min_per_km"] <= 8)
].copy()

df_easy["speed_m_per_min"] = (df_easy["distance_km"] * 1000) / (df_easy["duration_sec"] / 60)
df_easy["hr_efficiency"] = df_easy["speed_m_per_min"] / df_easy["avg_hr"]
df_easy = df_easy[df_easy["hr_efficiency"] < 1.5].copy()

# Aggregate to weekly

df_easy["week"] = df_easy["date"].dt.to_period("W").dt.to_timestamp()

weekly = df_easy.groupby("week").agg(
    avg_pace=("pace_min_per_km", "mean"),
    avg_hr_efficiency=("hr_efficiency", "mean"),
    avg_hr=("avg_hr", "mean"),
    total_km=("distance_km", "sum"),
    n_runs=("distance_km", "count"),
    avg_elevation=("elevation_m", "mean"),
).reset_index()

print(f"Weekly samples before merge: {len(weekly)}")
print(weekly.head())

# Fix daily weekly aggregation to match
df_daily = df.set_index("date")[["distance_km"]].resample("D").sum().fillna(0)
df_daily["rolling_28d"] = df_daily["distance_km"].rolling(28).sum()
df_daily["rolling_42d"] = df_daily["distance_km"].rolling(42).sum()

# Use W-MON to match to_period("W") which starts on Monday
df_daily_weekly = df_daily.resample("W-MON").mean().reset_index()
df_daily_weekly.columns = ["week", "daily_km", "rolling_28d", "rolling_42d"]
df_daily_weekly["week"] = pd.to_datetime(df_daily_weekly["week"])

print("weekly sample:", weekly["week"].iloc[0])
print("df_daily_weekly sample:", df_daily_weekly["week"].iloc[0])

weekly = weekly.merge(df_daily_weekly, on="week", how="left")
print(f"Weekly samples after merge: {len(weekly)}")

Weekly samples before merge: 67
        week  avg_pace  avg_hr_efficiency  avg_hr  total_km  n_runs  \
0 2024-07-08  7.203622           1.030845  135.55   11.1928       2   
1 2024-07-29  6.582714           1.070023  143.38   23.6540       5   
2 2024-08-26  5.697239           1.254461  142.75   11.4640       2   
3 2024-09-02  6.083203           1.170015  140.50    7.1207       1   
4 2024-09-09  6.031780           1.206937  141.05   21.5669       4   

   avg_elevation  
0            0.0  
1            2.0  
2            0.5  
3           34.0  
4           19.0  
weekly sample: 2024-07-08 00:00:00
df_daily_weekly sample: 2021-04-26 00:00:00
Weekly samples after merge: 67


In [8]:
# Target: NEXT week's avg pace
weekly["next_week_pace"] = weekly["avg_pace"].shift(-1)
weekly = weekly.dropna().reset_index(drop=True)

print(f"Final samples for training: {len(weekly)}")

features = [
    "avg_hr_efficiency",
    "avg_hr",
    "total_km",
    "n_runs",
    "rolling_28d",
    "rolling_42d",
    "avg_elevation"
]

X = weekly[features]
y = weekly["next_week_pace"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model2 = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

model2.fit(X_train, y_train)
y_pred = model2.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.3f} min/km ({mae*60:.1f} sec/km)")
print(f"R²: {r2:.3f}")

Final samples for training: 66
MAE: 0.366 min/km (22.0 sec/km)
R²: 0.011


## Model 6 — Effort Classification (K-Means on Split Data)
Automatically discovering run types from km-by-km pace and HR patterns.
No manual labels needed — unsupervised clustering finds natural groupings.
Similar approach to the AVV cycling project (K-Means on station archetypes).

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load splits
splits = pd.read_csv("../data/raw/splits.csv")
print(f"Split rows: {len(splits)}")
print(f"Activities: {splits['activity_id'].nunique()}")
splits.head()

Split rows: 4187
Activities: 480


,activity_id,split,distance_m,elapsed_time_sec,moving_time_sec,elevation_difference,avg_hr,avg_speed,pace_min_per_km
0,5159424797,1,1001.6,302,302,-0.1,NaN,3.32,5.025293
1,5159424797,2,998.8,305,305,-0.1,NaN,3.27,5.089441
2,5159424797,3,1002.4,314,314,0.0,NaN,3.19,5.220803
3,5159424797,4,13.5,4,4,0.0,NaN,3.38,4.938272
4,5207562011,1,1002.7,299,299,-0.1,NaN,3.35,4.969915


In [10]:
# Aggregate splits to activity level — capture the shape of each run
activity_features = splits.groupby("activity_id").agg(
    avg_pace=("pace_min_per_km", "mean"),
    pace_std=("pace_min_per_km", "std"),        # pace variability
    pace_first_half=("pace_min_per_km", lambda x: x.iloc[:len(x)//2].mean()),
    pace_second_half=("pace_min_per_km", lambda x: x.iloc[len(x)//2:].mean()),
    avg_hr=("avg_hr", "mean"),
    hr_std=("avg_hr", "std"),
    n_splits=("split", "count"),
    total_distance=("distance_m", "sum"),
    elevation_total=("elevation_difference", "sum")
).reset_index()

# Pace split ratio — negative split = faster in second half
activity_features["split_ratio"] = (
    activity_features["pace_second_half"] / activity_features["pace_first_half"]
)

# Drop rows with missing values
activity_features = activity_features.dropna().reset_index(drop=True)
print(f"Activities with full split data: {len(activity_features)}")
activity_features.head()

Activities with full split data: 315


,activity_id,avg_pace,pace_std,pace_first_half,pace_second_half,avg_hr,hr_std,n_splits,total_distance,elevation_total,split_ratio
0,10879633356,4.849815,0.180325,4.756690,4.927419,-0.888497,0.369815,11,10134.4,0.0,1.035892
1,11417994826,4.226158,0.125627,4.269215,4.191712,112.837100,8.286562,9,8033.3,0.0,0.981846
2,11432342451,4.115942,0.386431,4.177661,4.054223,0.765000,3.006008,8,7238.7,0.0,0.970453
3,11692333603,4.332407,0.262893,4.518819,4.169296,49.781574,25.985626,15,14043.8,0.0,0.922652
4,11845159498,6.713149,1.286775,6.068003,7.358295,143.729994,8.756430,6,5145.2,0.0,1.212639


In [11]:
features_for_clustering = [
    "avg_pace",
    "pace_std",
    "split_ratio",
    "avg_hr",
    "hr_std",
    "n_splits",
    "elevation_total"
]

X = activity_features[features_for_clustering].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Find optimal k with elbow method
inertias = []
k_range = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig = px.line(x=list(k_range), y=inertias,
              title="Elbow Method — Optimal Number of Clusters",
              labels={"x": "Number of clusters", "y": "Inertia"})
fig.show()

In [12]:
# Fit with k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
activity_features["cluster"] = km.fit_predict(X_scaled)

# Label clusters based on characteristics
cluster_means = activity_features.groupby("cluster")[features_for_clustering].mean()
print(cluster_means)

         avg_pace  pace_std  split_ratio      avg_hr     hr_std   n_splits  \
cluster                                                                      
0        7.562400  4.172695     1.337992  153.535228  11.653763   7.200000   
1        5.208777  0.505251     0.954238  153.549148  11.677587  10.570312   
2        5.470626  0.476350     1.053479  156.122006   9.166213  11.481481   
3        6.054990  0.399372     1.020330  137.976471   5.501387   8.406452   

         elevation_total  
cluster                   
0              17.720000  
1               0.389062  
2             -36.200000  
3               1.512258  


In [13]:
cluster_labels = {
    0: "Walk/Hike",
    1: "Tempo/Race",
    2: "Hilly Run",
    3: "Easy Run"
}

activity_features["run_type"] = activity_features["cluster"].map(cluster_labels)

# Count per type
print(activity_features["run_type"].value_counts())

run_type
Easy Run      155
Tempo/Race    128
Hilly Run      27
Walk/Hike       5
Name: count, dtype: int64


In [14]:
# PCA for 2D visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

activity_features["pca1"] = X_pca[:, 0]
activity_features["pca2"] = X_pca[:, 1]

fig = px.scatter(activity_features, x="pca1", y="pca2",
                 color="run_type",
                 title="Run Type Clusters (PCA projection)",
                 labels={"pca1": "PC1", "pca2": "PC2"})
fig.show()

# Distribution of run types
fig2 = px.bar(activity_features["run_type"].value_counts().reset_index(),
              x="run_type", y="count",
              title="Distribution of Run Types",
              labels={"run_type": "Run Type", "count": "Count"})
fig2.show()